
<div align="center">

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiangtgao/DS-UA_201-Causal-Inference-Spring-2025/blob/main/labs/10-Matching_Instruments.ipynb)

</div>

$$
\begin{array}{c}
\textbf{CAUSAL INFERENCE -- Matching}\\\\
\textbf{Hamza Alshamy} \\
\textit{Center for Data Science, New York University} \\\\
\textit{Oct 31, 2025}\\\\\\
\end{array}
$$

---

## Goals For Today

1. Recap
2. Matching
    - Exact Matching
    - Inexact Matching (e.g., Nearest-Neighbor Matching)

**Recap**

$$
Y = \underbrace{\beta_0 + \beta_1 \cdot X_1}_{\hat{Y}} + U
$$

The idea is that $U$ may contain several components that affect $Y$ besides $X_1$.

$$
U = \underbrace{(C_1 + C_2)}_{\text{Observable}} + \underbrace{U^*}_{\text{What remains unobservable}}
$$

Some of these components may be **observable**, while others remain **unobservable**.

Suppose we can observe one of these components — call it $C$.  
We can then control for it by including it in the regression model:

$$
Y = \beta_0 + \beta_1 \cdot S + \beta_2 \cdot C + U
$$

If $C$ successfully captures the factor that jointly influences $S$ and $Y$, then we can assume that $U$ is independent of $S$ given $C$, that is:

$$
U \perp S \mid C
$$

**Illustration: The Child Adoption Example**

- **Set-up:**  
  Researchers want to estimate the causal effect of **parental wealth** ($S$) on **child wealth** ($Y$).

- **Problem:**  
  Parental wealth is correlated with unobserved factors — such as **genetic traits** that influence both the parents’ ability to accumulate wealth and the child’s future income. In other words: 

  $$
  S \not \perp U
  $$

- **Context:**  
  In the **Korean adoption program** in Norway, children were matched to adoptive parents **in the order that applications were approved**.  
  This process limited parents’ ability to select specific children, creating near-random assignment conditional on application timing.

- **Identification strategy:**  
  By controlling for the **year (or timing) of application** ($C$), we account for the only systematic factor that could affect both the match and child outcomes. Within each year, which child is paired with which family is effectively random.

- **Key assumption:**  
  $$
  S \perp U \mid C
  $$  
  Once we condition on the year of application, parental wealth is independent of all other unobserved influences on child outcomes.


- **Interpretation:**  
  The regression coefficient on $S$ from  
  $$
  Y = \beta_0 + \beta_1 \cdot S + \beta_2 \cdot C_{1965} + \beta_3 \cdot C_{1966} + \beta_4 \cdot C_{1967}  + U
  $$ 
  ,where each $C_t$ is a **year** indicator variable for year $t$, can then be interpreted as the **causal effect** of parental wealth on child wealth.

- **Why use indicator (dummy) variables for each year instead of a single numeric year variable?**  
  The identification strategy requires assuming that  
  $$
  E[U \mid C] \text{ is linear in } C.
  $$  
  This assumption is **automatically satisfied** when $C$ is represented through 0–1 year indicators, because each indicator is binary.  
  If instead we used a single numeric year variable (e.g., 1965, 1966, 1967), we would be **forcing** $E[U \mid C]$ to follow a *linear trend over years*, which is unlikely and could violate the assumption.  
  Using year dummies avoids imposing any functional-form restrictions on how $U$ varies across years.


**Caveat**

- Controlling for $C$ only accounts for the **observable** component of $U$.  
- If there are unobservable factors that still correlate with both $S$ and $Y$,the estimate of $\beta_1$ will remain biased.  

As the researcher, it is your responsibility to justify *why* conditioning on $C$ plausibly makes $S$ independent of $U$.

# Matching

## Exact Matching

- Intuitively, $S ⊥ U|C$ implies that within each sub-population, $S$ is as good as random
- Then, within any sub-population, direct comparison of the treated and untreated groups should estimate the causal effect of $S$
- To then get at the overall ATE, we just aggregate across the effects within each group

To obtain treated and control groups with similar covariate distributions.

We have multiple subgroups based on the control variables and $S \perp U | C$, we can use the matching estimator:

$$
\begin{aligned}
\operatorname{Matching}
& =  \sum_{c \in C} \left( \mathbb{E}[Y \mid S=1, C = c] - \mathbb{E}[Y \mid S=0, C = c] \right) \cdot \mathbb{P}(C = c)
\end{aligned}
$$

We can look at the example from last time:

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as sm

In [2]:
# Step 1: Simulate family status (rich=1, poor=0)
np.random.seed(42)
n = 1000
family_status = np.random.binomial(1, 0.4, n)  # 40% rich, 60% poor

# Step 2: Simulate college attendance based on family status (rich more likely to attend college)
college = np.random.binomial(1, 0.7 * family_status + 0.3 * (1 - family_status), n)

# Step 3: Simulate income based on family status and college attendance

noise = np.random.normal(0, 5000, n)  # Noise to add to income
income_observe = (50000 + 30000 * college + 20000 * family_status + noise)
income_college = (50000 + 30000 * 1 + 20000 * family_status + noise)

# For people who went to college (treated group), compared to the untreated,
# they would have higher income in the counterfactual world where they did not go to college
income_no_college = (50000 + 30000 * 0 + 10000 * college + 20000 * family_status + noise)

# Create a DataFrame for clarity
df = pd.DataFrame({
    'Family_Status': family_status,  # 1 = rich, 0 = poor
    'College': college,  # 1 = went to college, 0 = did not go to college
    'Income_Observe': income_observe,
    'Income_College': income_college,
    'Income_No_College': income_no_college
})

In [3]:
df.head()

,Family_Status,College,Income_Observe,Income_College,Income_No_College
0,0,0,45610.087066,75610.087066,45610.087066
1,1,1,95865.598230,95865.598230,75865.598230
2,1,0,68867.605540,98867.605540,68867.605540
3,0,1,81836.827534,81836.827534,61836.827534
4,0,1,84567.923131,84567.923131,64567.923131


Using our causal model notation:

- **Average Treatment Effect (ATE)**  
  The average causal effect of treatment across the entire population:
  $$
  \text{ATE} = \mathbb{E}[\,Y(1,U) - Y(0,U)\,]
  $$

- **Average Treatment Effect on the Treated (ATT)**  
  The average causal effect for individuals who actually received the treatment ($S=1$):
  $$
  \text{ATT} = \mathbb{E}[\,Y(1,U) - Y(0,U) \mid S = 1\,]
  $$

- **Average Treatment Effect on the Untreated (ATU)**  
  The average causal effect for individuals who did *not* receive the treatment ($S=0$):
  $$
  \text{ATU} = \mathbb{E}[\,Y(1,U) - Y(0,U) \mid S = 0\,]
  $$

In the code below, we estimate these quantities using the simulated potential outcomes:  
`Income_College` corresponds to $Y(1,U)$ and `Income_No_College` corresponds to $Y(0,U)$.


In [28]:
# Step 4: Compute ATE, ATT, ATU
# TRUE ATE:
ATE = df['Income_College'].mean() - df['Income_No_College'].mean()

# ATT:
ATT = df[(df['College'] == 1)]['Income_College'].mean() - df[(df['College'] == 1)]['Income_No_College'].mean()

# ATU:
ATU = df[(df['College'] == 0)]['Income_College'].mean() - df[(df['College'] == 0)]['Income_No_College'].mean()

# Print the results
print(f"TRUE Average Treatment Effect (ATE): {ATE:.2f}")
print(f"TRUE Average Treatment Effect on the Treated (ATT): {ATT:.2f}")
print(f"TRUE Average Treatment Effect on the Untreated (ATU): {ATU:.2f}")

TRUE Average Treatment Effect (ATE): 25560.00
TRUE Average Treatment Effect on the Treated (ATT): 20000.00
TRUE Average Treatment Effect on the Untreated (ATU): 30000.00


In [29]:
# Step 5: Compute association by conditioning
# Biased ATE using only what was obserevd:
Association = df[df['College'] == 1]['Income_Observe'].mean() - df[df['College'] == 0]['Income_Observe'].mean()

# Print the results
print(f"Biased Average Treatment Effect (ATE): {Association:.2f}")

Biased Average Treatment Effect (ATE): 37639.89


In [4]:
# Step 6: Condition on confounder - family status

ATE_family_1 = df[(df['College'] == 1) & (df['Family_Status'] == 1)]['Income_Observe'].mean() - df[(df['College'] == 0) & (df['Family_Status'] == 1)]['Income_Observe'].mean()

# Print the results
print(f"ATE for those with family_status = 1: {ATE_family_1:.2f}")

ATE_family_0 = df[(df['College'] == 1) & (df['Family_Status'] == 0)]['Income_Observe'].mean() - df[(df['College'] == 0) & (df['Family_Status'] == 0)]['Income_Observe'].mean()

# Print the results
print(f"ATE for those with family_status = 0: {ATE_family_0:.2f}")

ATE for those with family_status = 1: 30553.75
ATE for those with family_status = 0: 30308.08


In [5]:
proportions = df['Family_Status'].value_counts()/len(df)
proportions

Family_Status
0    0.613
1    0.387
Name: count, dtype: float64

Under the assumption of conditional independence ($S \perp U \mid C$), we can express the **matching estimator** for the Average Treatment Effect (ATE) as:

$$
\widehat{\text{ATE}}_{\text{match}}
= \sum_{c \in \mathcal{C}}
\Big(\,\widehat{\mathbb{E}}[Y \mid S=1, C=c]
- \widehat{\mathbb{E}}[Y \mid S=0, C=c]\,\Big)
\cdot \widehat{\mathbb{P}}(C=c)
$$

In our example, the control variable is **Family Status** ($C$), so we have two subgroups: $C=1$ (rich) and $C=0$ (poor). The sample implementation becomes:

$$
\widehat{\text{ATE}}_{\text{match}}
= \widehat{\text{ATE}}_{C=1} \cdot \widehat{P}(C=1)
+ \widehat{\text{ATE}}_{C=0} \cdot \widehat{P}(C=0)
$$

where:
- $\widehat{\text{ATE}}_{C=1}$ is the estimated treatment effect among rich families,  
- $\widehat{\text{ATE}}_{C=0}$ is the estimated treatment effect among poor families, and  
- $\widehat{P}(C=c)$ is the sample proportion of each subgroup.


In [6]:
# The matching estimator for ATE is
ATE_matching = ATE_family_1 * proportions[1] + ATE_family_0 * proportions[0]
print(f"Matching estimate for ATE: {ATE_matching:.2f}")

Matching estimate for ATE: 30403.15


## Inexact Matching and the Curse of Dimensionality (matching)

Suppose we are estimating the causal effect of **military service** ($S$) on **future income** ($Y$), and we are willing to believe that assignment to military service is conditionally independent of unobservables:

$$
S \perp U \mid \text{age, years of schooling, application year, AFQT score}
$$

This means that, after conditioning on these variables, military service is “as good as random.”  

However, because there are **many control variables**, each taking on **many possible values**,  
exact matching becomes impossible — we will not find treated and untreated individuals with *identical* values for all controls.  

For example:
- One individual might be 20 years old, have 12 years of schooling, apply in 1975, and score 83 on the AFQT.
- It is unlikely that there exists another individual with **exactly** those same characteristics who did *not* serve.

This is known as the **curse of dimensionality**: the number of distinct covariate combinations grows quickly, making exact matches rare or nonexistent.

In such situations, we use **inexact matching**, where we find individuals who are *similar* (but not identical) on the observed controls and compare their outcomes.

### **Nearest-Neighbor Matching (A form of inexact matching)**

When exact matching is infeasible due to continuous or high-dimensional controls, we can use **Nearest-Neighbor Matching (NNM)** — a form of *inexact matching* performed at the **individual level**.

**Idea:** for each treated individual ($S_i = 1$), find one or more untreated individuals ($S_j = 0$) whose control values ($C_j$) are **closest** to those of the treated unit.

Formally, for each treated observation $i$:
$$
j(i) = \arg\min_{j:S_j=0} D(C_i, C_j)
$$
where $D(C_i, C_j)$ measures the distance between two observations in the covariate space (e.g., Euclidean distance).

> For each treated individual $i$, find the untreated individual $j$ whose covariates are closest to $i$. That is, the $j$ that minimizes the distance between $C_i$ and $C_j$

The individual-level treatment effect is then estimated as:
$$
\widehat{\tau}_i = Y_i - Y_{j(i)}.
$$

Aggregating across all treated units gives the **Average Treatment Effect on the Treated (ATT)**:
$$
\widehat{ATT}_{NN}
= \frac{1}{N_T} \sum_{i:S_i=1}
\Big( Y_i(S_i=1, U_i) - Y_{j(i)}(S_{j(i)}=0, U_{j(i)}) \Big),
$$
where $N_T$ is the number of treated individuals.

**Key intuition:**  
Nearest-Neighbor Matching constructs a *counterfactual outcome* for each treated unit using the outcome of a similar untreated unit. By averaging these individual contrasts, we obtain a population-level causal estimate.